# Progetto Smart Vehicular Systems - Multi-modal black box recorder
  - Gabriele Arcese
  - Diego Colì

## Setup

In [ ]:
import carla
import time
import random
import threading
import queue
from dataclasses import dataclass, field, asdict
from typing import Optional
import numpy as np

try:
    import pygame
except ImportError:
    pygame = None
    
try:
    import cv2
    import numpy as np
except ImportError:
    cv2 = None
    np = None

## Connessione al client

In [ ]:
client = carla.Client("localhost", 2000)
client.set_timeout(15.0)
world = client.get_world()
spectator = world.get_spectator()

print(f"Client version: {client.get_client_version()}")
print(f"Server version: {client.get_server_version()}")

## Dataclass EventRecord
Prodotto ad ogni tick della simulazione: è l'unità atomica del ring buffer.

| Campo | Sorgente CARLA | Tipo |
|---|---|---|
| `timestamp_sim` | `world.get_snapshot().timestamp.elapsed_seconds` | `float` |
| `timestamp_wall` | `time.time()` | `float` |
| `frame_id` | contatore incrementale | `int` |
| `camera_frames` | callback `sensor.camera.rgb` (una per camera) | `dict[str, np.ndarray]` |
| `location` | `vehicle.get_location()` | `carla.Location` |
| `velocity` | `vehicle.get_velocity()` | `carla.Vector3D` |
| `acceleration` | `vehicle.get_acceleration()` | `carla.Vector3D` |
| `control` | `vehicle.get_control()` | `carla.VehicleControl` |
| `warnings` | logica interna (LiDAR, collisione) | `list[str]` |
| `mqtt_messages` | thread MQTT | `list[str]` |
| `reasoning_text` | explanation agent | `str \| None` |
| `reasoning_ts` | wall-clock emissione agent | `float \| None` |

In [ ]:
@dataclass
class VehicleState:
    """Stato cinematico e di controllo del veicolo in un istante."""
    x: float
    y: float
    z: float
    vx: float
    vy: float
    vz: float
    ax: float
    ay: float
    az: float
    throttle: float
    brake: float
    steer: float
    hand_brake: bool
    reverse: bool

@property
def speed_ms(self) -> float:
    """Velocità scalare in m/s."""
    return float(np.sqrt(self.vx**2 + self.vy**2 + self.vz**2))

@property
def speed_kmh(self) -> float:
    return self.speed_ms * 3.6

@dataclass
class EventRecord:
    """Unità atomica del ring buffer — un record per tick di simulazione."""
    # --- Temporali ---
    frame_id: int            # contatore tick, usato come chiave nel viewer
    timestamp_sim: float     # secondi simulazione (univoco per tick in sync mode)
    timestamp_wall: float    # wall-clock al momento della costruzione

    # --- Percezione ---
    vehicle_state: VehicleState

    # --- Camera (dict role → frame BGR; vuoto se nessun frame è ancora arrivato) ---
    # Esempio: {"front": np.ndarray, "rear": np.ndarray}
    camera_frames: dict = field(default_factory=dict, repr=False)

    # --- Logica e messaggistica ---
    warnings: list = field(default_factory=list)       # es. ["COLLISION_IMMINENT"]
    mqtt_messages: list = field(default_factory=list)  # payload raw ricevuti nel tick

    # --- Explanation agent ---
    reasoning_text: Optional[str] = None
    reasoning_ts: Optional[float] = None  # wall-clock di emissione dell'agente

def to_dict(self) -> dict:
    """Serializza tutto tranne camera_frames (salvati separatamente come JPEG)."""
    d = asdict(self)
    d.pop("camera_frames")  # i frame vanno in frames/<frame_id:06d>_<role>.jpg
    return d

def has_camera(self, role: str = None) -> bool:
    """True se almeno una camera (o la camera `role`) ha un frame nel tick."""
    if role:
        return role in self.camera_frames
    return len(self.camera_frames) > 0

def camera_roles(self) -> list:
    return list(self.camera_frames.keys())

def has_reasoning(self) -> bool:
    return self.reasoning_text is not None

### Helper
Costruisce VehicleState da un attore.

In [ ]:
def build_vehicle_state(vehicle) -> VehicleState:
    loc = vehicle.get_location()
    vel = vehicle.get_velocity()
    acc = vehicle.get_acceleration()
    ctrl = vehicle.get_control()
    return VehicleState(
    x=loc.x, y=loc.y, z=loc.z,
    vx=vel.x, vy=vel.y, vz=vel.z,
    ax=acc.x, ay=acc.y, az=acc.z,
    throttle=ctrl.throttle,
    brake=ctrl.brake,
    steer=ctrl.steer,
    hand_brake=ctrl.hand_brake,
    reverse=ctrl.reverse,
)